In [10]:
import os
import gc
from tqdm import tqdm
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoConfig
from transformers import DataCollatorForLanguageModeling
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

import evaluate
import datasets
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict, load_metric

import warnings
warnings.filterwarnings('ignore')

In [11]:
class CFG:
    wandb = False
    report_to = None
    lab_assignment = 5
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    tokenizer_name = 'bayartsogt/mongolian-gpt2'
    model_name = 'bayartsogt/mongolian-gpt2'

    project = 'NUM-Machine-Learning-Lab-5'
    name = "Lab 5"

    config = {
        "output_dir": "lab5_finetune_gpt2",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-2,
        'num_train_epochs': 3,
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "dataloader_num_workers": num_workers,
        "finetuning_task": 'ner',
        "evaluation_strategy": 'epoch',
        "logging_strategy": 'epoch',
        "overwrite_output_dir": True,
        "push_to_hub": False,
    }

    model_save_dir = "lab5_gpt2"

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "seqeval"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

In [12]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

In [13]:
df = pd.read_csv('khk.noun.tsv', sep = '\t')
df['length_greater'] = df.apply(lambda x: len(x['prompt']) > len(x['answer'].split(' ')[0]), axis = 1)
df['length_equal'] = df.apply(lambda x: len(x['prompt']) == len(x['answer'].split(' ')[0]), axis = 1)
df[~(df['length_equal'] | df['length_greater'])]

,prompt,answer,length_greater,length_equal
315,хэвээ,хэвийн <voc>,False,False
398,урагтай,ургийнхан <com>,False,False
982,дуранд,дураараа <dat>,False,False
997,ааваа,аавынх <voc>,False,False
1124,дүрмээ,дүрмийн <voc>,False,False
1777,ургаар,ургийнхан <ins>,False,False
2369,ургийн,ургийнхан <gen>,False,False
2574,хэвд,хэвийн <dat>,False,False
2781,ургаа,ургийнхан <voc>,False,False
3068,сурталчилгааг,сурталчилгааны <acc>,False,False


In [14]:
# Энэ хэд буруу юм шиг харагдаад байгаа болохоор хаячихая
df = df[df['length_equal'] | df['length_greater']].reset_index(drop = True)
df['answer_text'] = df['answer'].apply(lambda x: x.split(' ')[0].strip())
df['answer_something'] = df['answer'].apply(lambda x: x.split(' ')[1].strip())
df[df['length_equal']]

,prompt,answer,length_greater,length_equal,answer_text,answer_something
28,махчны,махчин <gen>,False,True,махчин,<gen>
366,ажилтны,ажилтан <gen>,False,True,ажилтан,<gen>
399,хүчийг,хүчээр <acc>,False,True,хүчээр,<acc>
868,оюутны,оюутан <gen>,False,True,оюутан,<gen>
930,байнгын,байнгын <gen>,False,True,байнгын,<gen>
...,...,...,...,...,...,...
13494,ордны,ордон <gen>,False,True,ордон,<gen>
13563,жуулчны,жуулчин <gen>,False,True,жуулчин,<gen>
13609,тамирчны,тамирчин <gen>,False,True,тамирчин,<gen>
13698,сонирхогчийн,сонирхогчийн <gen>,False,True,сонирхогчийн,<gen>


In [15]:
def longest_common_sub(tmp_1, tmp_2):
  matcher = SequenceMatcher(None, tmp_1, tmp_2)
  longest_match = matcher.find_longest_match()
  return tmp_1[longest_match.a + longest_match.size - longest_match.a: ]

print(longest_common_sub('багтраагаа', 'багтраа'))
print(longest_common_sub('махчны', 'махчин'))
print(longest_common_sub('янхны', 'янхан'))
print(longest_common_sub('тамирчны', 'тамирчин'))

гаа
ны
ны
ны


In [16]:
def longest_common_sub(row):
    tmp_1 = row['prompt']
    tmp_2 = row['answer_text']
    something = row['answer_something']

    matcher = SequenceMatcher(None, tmp_1, tmp_2)
    longest_match = matcher.find_longest_match()
    return tmp_1[longest_match.a + longest_match.size - longest_match.a:] + ' ' + something

df['answer_2'] = df[['prompt', 'answer_text', 'answer_something']].apply(longest_common_sub, axis = 1)
df

,prompt,answer,length_greater,length_equal,answer_text,answer_something,answer_2
0,багтраагаа,багтраа <voc>,True,False,багтраа,<voc>,гаа <voc>
1,дэнсээс,дэнс <abl>,True,False,дэнс,<abl>,ээс <abl>
2,тархалтыг,тархалт <acc>,True,False,тархалт,<acc>,ыг <acc>
3,дуулийг,дууль <acc>,True,False,дууль,<acc>,ийг <acc>
4,гуалингаар,гуалин <ins>,True,False,гуалин,<ins>,гаар <ins>
...,...,...,...,...,...,...,...
14364,үтрээний,үтрээ <gen>,True,False,үтрээ,<gen>,ний <gen>
14365,лавлагаанд,лавлагаа <dat>,True,False,лавлагаа,<dat>,нд <dat>
14366,эзэмшлээ,эзэмшил <voc>,True,False,эзэмшил,<voc>,лээ <voc>
14367,улстай,улс <com>,True,False,улс,<com>,тай <com>


## Энд нэг жижиг асуудал байгаа

Жишээ нь 14368 дах индекс дээр `ар <ins>` буруу байгаа. Энд бид зүгээр шууд `ар`->`аар`, `эр`->`ээр`, `ор`->`оор` гэх мэтээр map хийж болох юм. Өөр санаа байна уу? Аан бас энэнээс өөр иймэрхүү асуудал гарах уу?

In [17]:
df['answer'] = df['answer_2']
df.drop(columns = ['answer_2', 'length_greater', 'length_equal', 'answer_text', 'answer_something'], inplace = True)

df['prompt'] = '<s> bb: ' + df['prompt']
df['answer'] = df['answer'] + '</s>'

infl_dataset = Dataset.from_pandas(df)
ds_train_devtest = infl_dataset.train_test_split(test_size = 0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size = 0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

ds_splits = ds_splits.flatten()

Map:   0%|          | 0/14009 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [19]:
block_size = 64
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)

def preprocess_function(examples):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = ds_splits["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/14009 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [20]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_ds.map(group_texts, batched = True, num_proc = 4)

Map (num_proc=4):   0%|          | 0/14009 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [21]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False)
model = AutoModelForCausalLM.from_pretrained(CFG.model_name)

config = CFG.config

test_ver = 1
OUTPUT_MODEL = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")

training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = OUTPUT_MODEL,
    num_train_epochs = config["num_train_epochs"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    evaluation_strategy = config["evaluation_strategy"],
    push_to_hub = config["push_to_hub"],
    do_eval = True,
    disable_tqdm = True
)

In [22]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset["train"],
    eval_dataset = lm_dataset["valid"],
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = None,
)

trainer.train()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


{'eval_loss': 1.4474750757217407, 'eval_runtime': 0.0823, 'eval_samples_per_second': 388.868, 'eval_steps_per_second': 48.608, 'epoch': 1.0}
{'loss': 2.4655, 'grad_norm': 1.0944838523864746, 'learning_rate': 9.583333333333335e-06, 'epoch': 1.56}
{'eval_loss': 1.2923219203948975, 'eval_runtime': 0.0765, 'eval_samples_per_second': 418.059, 'eval_steps_per_second': 52.257, 'epoch': 2.0}
{'eval_loss': 1.2711377143859863, 'eval_runtime': 0.0755, 'eval_samples_per_second': 423.622, 'eval_steps_per_second': 52.953, 'epoch': 3.0}
{'train_runtime': 109.6643, 'train_samples_per_second': 69.922, 'train_steps_per_second': 8.754, 'train_loss': 1.8779552459716797, 'epoch': 3.0}


TrainOutput(global_step=960, training_loss=1.8779552459716797, metrics={'train_runtime': 109.6643, 'train_samples_per_second': 69.922, 'train_steps_per_second': 8.754, 'train_loss': 1.8779552459716797, 'epoch': 3.0})

In [23]:
generator = pipeline("text-generation", model = model, tokenizer = tokenizer, num_beams = 5)

prompt = "<s> bb: гүйцэтгэлийг"
ans = generator(prompt, max_length = 20)

print(str(ans[0]['generated_text']))
print(str(ans))

gc.collect()

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


<s> bb: гүйцэтгэлийг ийг <acc> bb: довтой той <com
[{'generated_text': '<s> bb: гүйцэтгэлийг ийг <acc> bb: довтой той <com'}]


101

In [24]:
torch.cuda.empty_cache()
gc.collect()

del trainer
gc.collect()

0